In [13]:
# Load dataset
import pandas as pd
import ast
import re
from pathlib import Path

INPUT_FILE = Path(
    r"C:\Users\Visakh H S\Labour_Market_Intelligence\data\processed\final_labour_market_dataset.csv"
)

OUTPUT_FILE = Path(
    r"C:\Users\Visakh H S\Labour_Market_Intelligence\data\processed\final_labour_market_normalized.csv"
)

df = pd.read_csv(INPUT_FILE)

print("=" * 70)
print("STAGE 3 — FINAL DATASET LOADED")
print("=" * 70)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print(df.columns.tolist())

STAGE 3 — FINAL DATASET LOADED
Rows: 19207
Columns: 29
['job_id', 'source_id', 'title', 'company', 'location', 'description', 'salary_min', 'salary_max', 'salary_avg', 'created', 'contract_type', 'contract_time', 'url', 'source', 'title_clean', 'description_clean', 'industry', 'industry_original', 'second_pass_industry', 'industry_before_second_pass', 'technical_skills', 'soft_skills', 'tools_and_technologies', 'certifications', 'licenses', 'education_required', 'experience_required', 'languages', 'status']


In [3]:
# convert LLM fields
LIST_COLUMNS = [
    "technical_skills",
    "soft_skills",
    "tools_and_technologies",
    "certifications",
    "licenses",
    "languages"
]

def parse_list(value):

    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    value = str(value).strip()

    if not value:
        return []

    try:
        result = ast.literal_eval(value)

        if isinstance(result, list):
            return result

        return [str(result)]

    except (ValueError, SyntaxError):
        return [value]


for col in LIST_COLUMNS:
    df[col] = df[col].apply(parse_list)

print("List conversion complete.")

for col in LIST_COLUMNS:
    print(
        col,
        "→",
        df[col].apply(lambda x: isinstance(x, list)).all()
    )

List conversion complete.
technical_skills → True
soft_skills → True
tools_and_technologies → True
certifications → True
licenses → True
languages → True


In [4]:
# cleaning
def clean_text_value(value):

    if value is None:
        return ""

    value = str(value).strip()
    value = re.sub(r"\s+", " ", value)

    return value


def clean_list(items):

    cleaned = []

    for item in items:

        item = clean_text_value(item)

        if item:
            cleaned.append(item)

    return cleaned


for col in LIST_COLUMNS:
    df[col] = df[col].apply(clean_list)

print("Basic cleaning complete.")

Basic cleaning complete.


In [5]:
# Remove placeholders
PLACEHOLDERS = {
    "",
    "none",
    "none mentioned",
    "none mentioned in the job description",
    "not specified",
    "not specified in the job description",
    "unknown",
    "n/a",
    "na",
    "no specific tools mentioned",
    "no specific tools mentioned in the job description"
}

def remove_placeholders(items):

    cleaned = []

    for item in items:

        normalized = str(item).strip().lower()

        if normalized not in PLACEHOLDERS:
            cleaned.append(item)

    return cleaned


for col in LIST_COLUMNS:
    df[col] = df[col].apply(remove_placeholders)

print("Placeholder removal complete.")

Placeholder removal complete.


In [6]:
# normalization
NORMALIZATION_MAP = {

    "python programming": "Python",
    "python programming language": "Python",
    "python": "Python",

    "sql": "SQL",

    "java": "Java",

    "c#": "C#",
    "c++": "C++",

    "r programming": "R",
    "r language": "R",

    "html": "HTML",
    "html/css": "HTML/CSS",
    "css": "CSS",

    "javascript": "JavaScript",
    "java script": "JavaScript",

    "excel": "Microsoft Excel",
    "ms excel": "Microsoft Excel",
    "microsoft excel": "Microsoft Excel",

    "tableau": "Tableau",

    "aws": "AWS",
    "aws cloud": "AWS",

    "azure": "Azure",

    "apache spark": "Apache Spark",
    "spark": "Apache Spark",

    "apache hadoop": "Hadoop",
    "hadoop": "Hadoop",

    "apache kafka": "Apache Kafka",
    "kafka": "Apache Kafka",

    "docker": "Docker",

    "cad": "CAD",
    "auto cad": "AutoCAD",
    "autocad": "AutoCAD",

    "full uk driving licence": "Full UK Driving Licence",
    "full uk driving license": "Full UK Driving Licence",

    "driving licence": "Driving Licence",
    "driving license": "Driving Licence",

    "cscs": "CSCS Card",

    "sia licence": "SIA Licence",
    "sia license": "SIA Licence"
}


def normalize_skill(skill):

    skill = str(skill).strip()

    if not skill:
        return ""

    key = skill.lower()

    return NORMALIZATION_MAP.get(key, skill)


for col in LIST_COLUMNS:

    df[col + "_normalized"] = df[col].apply(
        lambda items: [
            normalize_skill(x)
            for x in items
        ]
    )

print("Skill normalisation complete.")

Skill normalisation complete.


In [7]:
# remove duplicates
NORMALIZED_COLUMNS = [
    "technical_skills_normalized",
    "soft_skills_normalized",
    "tools_and_technologies_normalized",
    "certifications_normalized",
    "licenses_normalized",
    "languages_normalized"
]


def remove_duplicates(items):

    result = []
    seen = set()

    for item in items:

        key = str(item).strip().lower()

        if key and key not in seen:
            result.append(item)
            seen.add(key)

    return result


for col in NORMALIZED_COLUMNS:
    df[col] = df[col].apply(remove_duplicates)

print("Duplicate skills removed.")

Duplicate skills removed.


In [8]:
# soft skills
INVALID_SOFT_SKILLS = {
    "physical",
    "job role",
    "salary",
    "salary package",
    "experience",
    "qualification"
}


def clean_soft_skills(items):

    return [
        item for item in items
        if item.lower().strip() not in INVALID_SOFT_SKILLS
    ]


df["soft_skills_normalized"] = (
    df["soft_skills_normalized"]
    .apply(clean_soft_skills)
)

print("Soft skills cleaned.")

Soft skills cleaned.


In [9]:
# languages
INVALID_LANGUAGES = {
    "philosophy",
    "physical",
    "communication",
    "customer service",
    "teamwork",
    "problem solving",
    "leadership"
}


def clean_languages(items):

    return [
        item for item in items
        if item.lower().strip() not in INVALID_LANGUAGES
    ]


df["languages_normalized"] = (
    df["languages_normalized"]
    .apply(clean_languages)
)

print("Languages cleaned.")

Languages cleaned.


In [10]:
# validation
print("=" * 70)
print("NORMALISATION VALIDATION")
print("=" * 70)

print("\nRows:", len(df))

for col in NORMALIZED_COLUMNS:

    coverage = df[col].apply(
        lambda x: len(x) > 0
    ).sum()

    percentage = coverage / len(df) * 100

    print(f"\n{col}")
    print(f"Jobs with values: {coverage:,}")
    print(f"Coverage: {percentage:.2f}%")
    

NORMALISATION VALIDATION

Rows: 19207

technical_skills_normalized
Jobs with values: 3,593
Coverage: 18.71%

soft_skills_normalized
Jobs with values: 6,424
Coverage: 33.45%

tools_and_technologies_normalized
Jobs with values: 779
Coverage: 4.06%

certifications_normalized
Jobs with values: 364
Coverage: 1.90%

licenses_normalized
Jobs with values: 528
Coverage: 2.75%

languages_normalized
Jobs with values: 200
Coverage: 1.04%


In [11]:
# inspect 20 jobs
display(
    df[
        [
            "title",
            "industry",
            "technical_skills",
            "technical_skills_normalized",
            "soft_skills",
            "soft_skills_normalized",
            "tools_and_technologies",
            "tools_and_technologies_normalized"
        ]
    ].sample(20, random_state=42)
)

,title,industry,technical_skills,technical_skills_normalized,soft_skills,soft_skills_normalized,tools_and_technologies,tools_and_technologies_normalized
3234,Assistant Management Accountant,Finance and Banking,[],[],"[Communication, Teamwork, Problem Solving, Att...","[Communication, Teamwork, Problem Solving, Att...",[],[]
9459,Technical Area Sales Manager (North West),Other,[],[],[],[],[],[]
14704,Business Development Manager,Other,[],[],[],[],[],[]
4347,Audit Senior,Finance and Banking,[],[],[],[],[],[]
1512,Semi-skilled operator,Other,[],[],[],[],[],[]
3528,Nursery Room Leader - Term Time,Healthcare,[],[],[],[],[],[]
10246,QA Assistant,Other,[],[],[Attention to Detail],[Attention to Detail],[],[]
9075,Sales Engineer - OEM,Engineering,[],[],[],[],[],[]
8980,Paralegal,Legal,[],[],[Attention to Detail],[Attention to Detail],[],[]
13541,Sales Development Representative,Other,[],[],[Communication],[Communication],"[Company phone, Laptop]","[Company phone, Laptop]"


In [12]:
# save
df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 70)
print("NORMALISED DATASET SAVED")
print("=" * 70)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Output:", OUTPUT_FILE)

NORMALISED DATASET SAVED
Rows: 19207
Columns: 35
Output: C:\Users\Visakh H S\Labour_Market_Intelligence\data\processed\final_labour_market_normalized.csv
